# EVA — Yandex Cloud
## 32 GB VRAM | 128-dim | 32 heads | 6 layers | ~1.2M params

### Подготовка (один раз)
1. `git clone https://github.com/BlackCatSpb/FCF.git && cd FCF`
2. `pip install -r requirements.txt`
3. Загрузить `connected_ru.npy` вручную в папку `real_data/`

In [ ]:
# 1. Setup
!pip install torch numpy scikit-learn loguru psutil -q 2>&1 | tail -1

import torch, os, sys
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name} | VRAM: {g.total_memory/1e9:.1f} GB")

In [ ]:
# 2. Verify data files
import numpy as np

# Find connected_ru.npy (check multiple locations)
candidates = [
    'real_data/connected_ru.npy',
    'connected_ru.npy',  # if uploaded to root
    'real_data/full_corpus_ids.npy',
]
npy_path = None
for c in candidates:
    if os.path.exists(c):
        npy_path = c
        break

if npy_path:
    data = np.load(npy_path, mmap_mode='r').astype(np.int32)
    print(f"Corpus: {npy_path} ({len(data)/1e6:.1f}M tokens)")
else:
    print("ERROR: connected_ru.npy not found!")
    print("Upload it to real_data/ or root folder.")
    sys.exit(1)

# Check checkpoints
ckpt_dir = 'checkpoints/symbolic'
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_files = os.listdir(ckpt_dir) if os.path.exists(ckpt_dir) else []
print(f"Checkpoints: {len(ckpt_files)} files")
if ckpt_files:
    for f in sorted(ckpt_files)[-5:]:
        print(f"  {f} ({os.path.getsize(os.path.join(ckpt_dir,f))/1e6:.1f}MB)")

In [ ]:
# 3. Run training (200K steps, ~6-8 hours on A100)
!python train_yandex.py

In [ ]:
# 4. Monitor (run while training)
with open('yandex_train_log.txt', 'r') as f:
    lines = f.readlines()
    for l in lines[-15:]:
        print(l.rstrip())

In [ ]:
# 5. Test generation
import torch, torch.nn.functional as F
from eva.symbolic.char_vocab import CharacterVocab
from eva.symbolic.unified_transformer import UnifiedMultidimensionalTransformer

cv = CharacterVocab(); DEVICE = 'cuda'
ut = UnifiedMultidimensionalTransformer(vocab_size=157, coord_dim=128,
    num_levels=8, scales_per_level=4, num_layers=6, d_ff=512).to(DEVICE)

ckpt = torch.load('checkpoints/symbolic/yandex_latest.pt', map_location='cpu')
ut.load_state_dict(ckpt['ut'], strict=False)
ut.eval()

# Load coordinates
ev = torch.load('checkpoints/symbolic/evolved_affinity.pt', map_location='cpu')
c = ev['coords'].to(DEVICE); c128 = torch.zeros(157, 128, device=DEVICE)
c128[:, :24] = c[:, :24]
g = torch.Generator(device=DEVICE).manual_seed(42)
c128[:, 24:] = torch.randn(157, 104, generator=g, device=DEVICE) * 0.02
c128 = c128 / c128.norm(dim=-1, keepdim=True).clamp(1e-8)
ut.set_symbol_coordinates(c128)

def gen(ids, n=30, T=0.8):
    ids = list(ids)
    with torch.no_grad():
        for _ in range(n):
            _, sc = ut(torch.tensor([ids], dtype=torch.long, device=DEVICE), return_scores=True)
            logits = sc[0, -1] / T
            sl, si = logits.sort(descending=True)
            cp = F.softmax(sl, dim=-1).cumsum(dim=-1)
            cut = (cp > 0.95).nonzero(as_tuple=True)[0]
            k = cut[0].item() + 1 if len(cut) > 0 else 30
            k = min(max(k, 3), 50)
            v, idx = logits.topk(k); p = F.softmax(v, dim=-1)
            for t in set(ids[-5:]):
                m = (idx == t).nonzero(as_tuple=True)[0]
                if len(m) > 0: p[m] *= 0.2
            p /= p.sum(); nt = idx[torch.multinomial(p, 1)].item()
            if nt <= 0 or nt >= 157: nt = idx[0].item()
            ids.append(nt)
    return ids

tests = ['привет','человек идет','солнце светит','сегодня хорошая','я люблю','метаданные хранят']
for w in tests:
    ids = cv.encode(w)[1:-1]
    if len(ids) >= 2:
        r = gen(ids, 30, 0.8)
        print(f"{w} -> {cv.decode(r)}")